# 第 7 周 – “价格合适”顶点项目

本周，我们微调了一个开源模型，以根据描述估算产品价格。

## 比赛顺序

- **第 1 天：** QLoRA  
- **第 2 天：** 提示数据和基础模型  
- **第 3 天：** 训练第 1 部分  
- **第 4 天：** 训练第 2 部分  
- **第 5 天：** 评估

## 第 2 天：提示数据和基础模型

从 HuggingFace 加载数据集，设置标记器，分析标记计数，并创建微调提示。

In [ ]:
# 将 week7 添加到路径中，以便我们可以导入定价器
import sys
from pathlib import Path
for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    week7_path = base / "week7"
    if (week7_path / "pricer").exists():
        sys.path.insert(0, str(week7_path))
        break

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
LITE_MODE = False  # Set True for smaller dataset

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("Warning: HF_TOKEN not set. Login manually with: login(token)")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel("Number of tokens in summary")
plt.ylabel("Count")
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
CUTOFF = 110
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With CUTOFF={CUTOFF}, we will truncate {cut:,} items which is {cut/len(items):.1%}")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
print("Sample summary:")
print(train[0].summary)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
for item in tqdm(train + val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
print("PROMPT:")
print(test[0].prompt)
print("\nCOMPLETION:")
print(test[0].completion)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
plt.figure(figsize=(15, 6))
plt.title(f"Prompt+Completion Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel("Number of tokens (prompt + completion)")
plt.ylabel("Count")
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()

In [ ]:
# 可选：将提示推送到您的 HuggingFace 帐户以进行培训
# 用户名 =“您的 hf 用户名”
# 数据集 = f"{用户名}/items_prompts_lite" 如果 LITE_MODE 否则 f"{用户名}/items_prompts_full"
# Item.push_prompts_to_hub（数据集，训练，val，测试）

# 或者使用 Ed 的公共数据集：
# https://huggingface.co/datasets/ed-donner/items_prompts_lite
# https://huggingface.co/datasets/ed-donner/items_prompts_full

## 第 3 天和第 4 天：使用 QLoRA 进行培训

使用 QLoRA（4 位量化 + LoRA）微调 Llama 3.2 3B。 **需要 GPU**（例如 Google Colab T4/A100）。

培训笔记本：
- [第 3 天和第 4 天 Colab](https://colab.research.google.com/drive/1fBTm_jzrFGr88PDOFTQF7JIlQ1JW15BG)
- [第 5 天评估 Colab](https://colab.research.google.com/drive/16e8aY_BlHjzzcR-2dCyDMCPdOQ8XeN1e)

下图：从 HuggingFace 加载提示并在本地运行训练（如果您有 GPU）。

In [ ]:
# 训练设置 - 在 Colab 或 GPU 机器上运行
# 取消注释并运行以加载提示数据集和训练

# 从数据集导入load_dataset
# 从变压器导入 AutoModelForCausalLM、AutoTokenizer、BitsAndBytesConfig
# 从peft导入LoraConfig，get_peft_model，prepare_model_for_kbit_training
# 从 trl 导入 SFTTrainer、SFTConfig
# 进口火炬

# PROMPTS_DATASET = "ed-donner/items_prompts_lite" # 或 items_prompts_full
# 数据集 = load_dataset(PROMPTS_DATASET)
# train_data = 数据集[“火车”]
# val_data = 数据集[“val”]

# def format_example(ex): return {"text": ex["prompt"] + ex["completion"]}
# train_formatted = train_data.map(format_example)
# val_formatted = val_data.map(format_example)

# # 加载具有 4 位量化的基础模型
# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
# 模型 = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
# tokenizer.pad_token = tokenizer.eos_token

# # LoRA 配置
# 模型=prepare_model_for_kbit_training（模型）
# lora_config = LoraConfig（r = 16，lora_alpha = 32，target_modules = [“q_proj”，“v_proj”]，lora_dropout = 0.05，task_type =“CAUSAL_LM”）
# 模型= get_peft_model（模型，lora_config）

# # SFTConfig + SFTTrainer - 然后 trainer.train()
print("See Colab links above for full training pipeline.")

## 第 5 天：评估

评估微调模型：加载它，对测试数据运行预测，计算平均绝对误差。

In [ ]:
# 使用 week7pricer.evaluator 进行评估（需要微调模型）
# 从 week7 目录运行或确保 week7 在路径上

try:
    from pricer.evaluator import evaluate, Tester
    def fine_tuned_pricer(item):
        # 替换为您的实际模型推理逻辑
        # 例如加载模型，分词器，在 item.test_prompt() 上运行生成
        return item.price  # placeholder
    fine_tuned_pricer.__name__ = "fine_tuned_llama"
    # 评估（fine_tuned_pricer，测试，大小=200）
    print("evaluate() and Tester available. Define your pricer and run evaluate(pricer, test).")
except ImportError as e:
    print(f"pricer.evaluator not found: {e}. Run from repo root or add week7 to path.")

## 结果：模型比较

跨基线、传统 ML、LLM 和微调模型的预测误差（平均绝对误差）。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import plotly.graph_objects as go

results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("GPT 4.1 Nano (Fine-tuned)", "skyblue", 75.91),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4 bit", "darkred", 110.72),
    ("Fine-tuned Lite", "red", 65.40),
    ("Fine-tuned Full", "red", 39.85)
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction error from each model",
    yaxis=dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width=1000,
    height=800
)

fig.show()